In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
# 创建 4 张灰度图片，每张图片大小是 28×28
x = torch.randn(4, 1, 28, 28)

print(x.shape)

torch.Size([4, 1, 28, 28])


In [3]:
conv = nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3
)

print(conv)

Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))


In [4]:
out = conv(x)

print(out.shape)

torch.Size([4, 8, 26, 26])


In [5]:
conv_same = nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3,
    padding=1
)

out_same = conv_same(x)

print(out_same.shape)

torch.Size([4, 8, 28, 28])


In [7]:
image = torch.tensor([
    [1, 1, 1, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1]
], dtype=torch.float32)

print(image)
print(image.shape)

tensor([[1., 1., 1., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 0., 0.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 1., 1.]])
torch.Size([5, 5])


In [8]:
image_4d = image.reshape(1, 1, 5, 5)

print(image_4d.shape)

torch.Size([1, 1, 5, 5])


In [9]:
kernel = torch.tensor([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
], dtype=torch.float32)

kernel_4d = kernel.reshape(1, 1, 3, 3)

print(kernel_4d.shape)

torch.Size([1, 1, 3, 3])


In [10]:
result = F.conv2d(image_4d, kernel_4d)

print(result)
print(result.shape)

tensor([[[[ 0.,  3.,  3.],
          [ 0.,  1.,  1.],
          [ 0., -1., -1.]]]])
torch.Size([1, 1, 3, 3])


In [11]:
conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)

x = torch.randn(4, 1, 28, 28)

out = conv(x)
out_relu = F.relu(out)

print(out.shape)
print(out_relu.shape)

torch.Size([4, 8, 28, 28])
torch.Size([4, 8, 28, 28])


In [12]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)

x = torch.randn(4, 8, 28, 28)

out = pool(x)

print(out.shape)

torch.Size([4, 8, 14, 14])


In [13]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=3,
            padding=1
        )
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=3,
            padding=1
        )
        
        self.fc = nn.Linear(16 * 7 * 7, 10)
        
    def forward(self, x):
        # 输入 x: [batch_size, 1, 28, 28]
        
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        # 现在 x: [batch_size, 8, 14, 14]
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        # 现在 x: [batch_size, 16, 7, 7]
        
        x = x.reshape(x.size(0), -1)
        # 现在 x: [batch_size, 16*7*7]
        
        x = self.fc(x)
        # 现在 x: [batch_size, 10]
        
        return x

In [14]:
model = SimpleCNN()

print(model)

SimpleCNN(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc): Linear(in_features=784, out_features=10, bias=True)
)


In [15]:
x = torch.randn(4, 1, 28, 28)

print("输入:", x.shape)

x = model.conv1(x)
print("conv1 后:", x.shape)

x = F.relu(x)
print("relu1 后:", x.shape)

x = model.pool(x)
print("pool1 后:", x.shape)

x = model.conv2(x)
print("conv2 后:", x.shape)

x = F.relu(x)
print("relu2 后:", x.shape)

x = model.pool(x)
print("pool2 后:", x.shape)

x = x.reshape(x.size(0), -1)
print("flatten 后:", x.shape)

x = model.fc(x)
print("fc 后:", x.shape)

输入: torch.Size([4, 1, 28, 28])
conv1 后: torch.Size([4, 8, 28, 28])
relu1 后: torch.Size([4, 8, 28, 28])
pool1 后: torch.Size([4, 8, 14, 14])
conv2 后: torch.Size([4, 16, 14, 14])
relu2 后: torch.Size([4, 16, 14, 14])
pool2 后: torch.Size([4, 16, 7, 7])
flatten 后: torch.Size([4, 784])
fc 后: torch.Size([4, 10])


In [16]:
model = SimpleCNN()

x = torch.randn(4, 1, 28, 28)

output = model(x)

print(output.shape)
print(output)

torch.Size([4, 10])
tensor([[-0.6279, -0.1180, -0.3498, -0.3771,  0.3349,  0.2589,  0.0673,  0.0061,
          0.2666, -0.1538],
        [-0.7303, -0.0849, -0.5408, -0.2985,  0.3361,  0.0551,  0.0774, -0.1941,
          0.2804, -0.0357],
        [-0.6255,  0.0794, -0.4123, -0.1513,  0.4899,  0.1237, -0.0040, -0.0689,
          0.1997, -0.0882],
        [-0.6869,  0.0024, -0.4987, -0.1865,  0.3506,  0.1229, -0.1273, -0.1735,
          0.2862, -0.0764]], grad_fn=<AddmmBackward0>)


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PracticeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 6, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 12, kernel_size=3, padding=1)
        self.fc = nn.Linear(12 * 7 * 7, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        return x


model = PracticeCNN()

x = torch.randn(8, 1, 28, 28)

output = model(x)

print("输入 shape:", x.shape)
print("输出 shape:", output.shape)
print(model)

输入 shape: torch.Size([8, 1, 28, 28])
输出 shape: torch.Size([8, 10])
PracticeCNN(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc): Linear(in_features=588, out_features=10, bias=True)
)
